# NHANES Data Preparation and BP Category Labelling

**Component 3: Hypertension Dietary Risk Assessment System**

This notebook prepares the dataset used to train the blood pressure (BP) category classifier. It merges five NHANES 2017-March 2020 files, applies the inclusion criteria, derives the BP category label, and constructs the dietary features used in the proposal.

Blood pressure readings are used only to derive the label and are excluded from the model inputs to avoid target leakage. Model training is carried out in the next notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests

MIN_AGE = 18               # minimum age for inclusion
RELIABLE_RECALL_CODE = 1   # dietary recall status code for a complete, reliable recall

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CDC_BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"
FILES = {
    "demo": "P_DEMO",      # demographics
    "body": "P_BMX",       # body measures
    "bp":   "P_BPXO",      # oscillometric blood pressure
    "day1": "P_DR1TOT",    # total nutrient intake, day 1
    "day2": "P_DR2TOT",    # total nutrient intake, day 2
}

## 1. Data loading

The files are downloaded from the CDC website on first use and cached locally in `data/raw/`, which is excluded from version control.

In [2]:
def load_nhanes(file_name):
    path = RAW_DIR / f"{file_name}.XPT"
    if not path.exists():
        response = requests.get(f"{CDC_BASE_URL}{file_name}.XPT",
                                headers={"User-Agent": "Mozilla/5.0"}, timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
    return pd.read_sas(path, format="xport")


raw = {key: load_nhanes(name) for key, name in FILES.items()}
for key, df in raw.items():
    print(f"{FILES[key]:<10} {df.shape[0]:>6} rows  {df.shape[1]:>4} columns")

P_DEMO      15560 rows    29 columns
P_BMX       14300 rows    22 columns
P_BPXO      11656 rows    12 columns
P_DR1TOT    14300 rows   168 columns
P_DR2TOT    14300 rows    85 columns


## 2. Variable selection and merging

| Group | Variables |
|---|---|
| Identifier | `SEQN` |
| Demographics | `RIDAGEYR` (age), `RIAGENDR` (sex) |
| Body measures | `BMXBMI` (BMI) |
| Blood pressure | `BPXOSY1-3` (systolic), `BPXODI1-3` (diastolic) |
| Recall status | `DR1DRSTZ`, `DR2DRSTZ` |
| Nutrients (day 1 / day 2) | sodium `DR1TSODI` / `DR2TSODI`, potassium `DR1TPOTA` / `DR2TPOTA`, energy `DR1TKCAL` / `DR2TKCAL`, saturated fat `DR1TSFAT` / `DR2TSFAT` |

Participants are linked using `SEQN`. Day 1 intake, demographics, body measures and blood pressure are combined with inner joins, so every participant has all of them. Day 2 is added with a left join because not all participants completed a second recall.

In [3]:
SBP_COLS = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
DBP_COLS = ["BPXODI1", "BPXODI2", "BPXODI3"]
DAY2_NUTRIENTS = ["DR2TSODI", "DR2TPOTA", "DR2TKCAL", "DR2TSFAT"]

demo = raw["demo"][["SEQN", "RIDAGEYR", "RIAGENDR"]]
body = raw["body"][["SEQN", "BMXBMI"]]
bp = raw["bp"][["SEQN"] + SBP_COLS + DBP_COLS]
day1 = raw["day1"][["SEQN", "DR1DRSTZ", "DR1TSODI", "DR1TPOTA", "DR1TKCAL", "DR1TSFAT"]]
day2 = raw["day2"][["SEQN", "DR2DRSTZ"] + DAY2_NUTRIENTS]

merged = (day1.merge(demo, on="SEQN")
              .merge(body, on="SEQN")
              .merge(bp, on="SEQN")
              .merge(day2, on="SEQN", how="left"))

assert merged["SEQN"].is_unique
print(f"Merged table: {merged.shape[0]} participants")

Merged table: 11656 participants


## 3. Inclusion criteria

- **Dietary recall:** participants are retained only if the day 1 recall is reliable (`DR1DRSTZ` = 1). Day 2 nutrient values are set to missing when the day 2 recall is absent or unreliable, but the participant is retained.
- **Age:** participants younger than `MIN_AGE` are excluded. Age is retained so that older-adult subsets can be evaluated separately.

In [4]:
print("Day 1 recall status:\n", merged["DR1DRSTZ"].value_counts(dropna=False).sort_index().to_string())

merged = merged[merged["DR1DRSTZ"] == RELIABLE_RECALL_CODE].copy()
merged.loc[merged["DR2DRSTZ"] != RELIABLE_RECALL_CODE, DAY2_NUTRIENTS] = np.nan
merged = merged[merged["RIDAGEYR"] >= MIN_AGE].copy()

print(f"\nParticipants after inclusion criteria: {len(merged)}")

Day 1 recall status:
 DR1DRSTZ
1.0    10434
2.0       95
5.0     1127

Participants after inclusion criteria: 8092


## 4. Blood pressure category

Systolic and diastolic pressure are each averaged over the available readings (missing readings are ignored). Participants with no valid reading are excluded. The category follows the thresholds defined in the proposal; where systolic and diastolic values fall in different categories, the higher category is assigned.

| Category | Systolic (mmHg) | | Diastolic (mmHg) |
|---|---|---|---|
| Normal | < 120 | and | < 80 |
| Elevated | 120-129 | and | < 80 |
| Stage 1 | 130-139 | or | 80-89 |
| Stage 2 | >= 140 | or | >= 90 |

In [5]:
merged["systolic_avg"] = merged[SBP_COLS].mean(axis=1)
merged["diastolic_avg"] = merged[DBP_COLS].mean(axis=1)
merged = merged.dropna(subset=["systolic_avg", "diastolic_avg"]).copy()

sbp, dbp = merged["systolic_avg"], merged["diastolic_avg"]
categories = ["Normal", "Elevated", "Stage 1", "Stage 2"]

# np.select returns the first matching condition, so the highest stage is tested first
merged["bp_category"] = pd.Categorical(
    np.select([(sbp >= 140) | (dbp >= 90),
               (sbp >= 130) | (dbp >= 80),
               (sbp >= 120)],
              ["Stage 2", "Stage 1", "Elevated"],
              default="Normal"),
    categories=categories, ordered=True)

counts = merged["bp_category"].value_counts().reindex(categories)
print(pd.DataFrame({"count": counts, "percent": (counts / counts.sum() * 100).round(1)}))

             count  percent
bp_category                
Normal        3303     43.9
Elevated      1057     14.0
Stage 1       1562     20.7
Stage 2       1607     21.3


## 5. Feature engineering

Variables are renamed for readability and sex is encoded as a binary indicator. The sodium-to-potassium (Na:K) ratio is computed where potassium intake is positive. Multi-day features are the mean of the available reliable recalls (one or two); the multi-day Na:K ratio is the ratio of mean sodium to mean potassium.

NHANES provides at most two 24-hour recalls per participant, so these multi-day features represent a two-day average, not a seven-day intake.

In [6]:
data = merged.rename(columns={
    "SEQN": "participant_id", "RIDAGEYR": "age", "BMXBMI": "bmi",
    "DR1TSODI": "sodium_mg_day1",    "DR2TSODI": "sodium_mg_day2",
    "DR1TPOTA": "potassium_mg_day1", "DR2TPOTA": "potassium_mg_day2",
    "DR1TKCAL": "energy_kcal_day1",  "DR2TKCAL": "energy_kcal_day2",
    "DR1TSFAT": "sat_fat_g_day1",    "DR2TSFAT": "sat_fat_g_day2",
}).copy()

data["sex"] = data["RIAGENDR"].map({1: "male", 2: "female"})
data["sex_female"] = (data["RIAGENDR"] == 2).astype(int)


def na_k_ratio(sodium, potassium):
    return sodium / potassium.where(potassium > 0)


data["na_k_ratio_day1"] = na_k_ratio(data["sodium_mg_day1"], data["potassium_mg_day1"])

for nutrient in ["sodium_mg", "potassium_mg", "energy_kcal", "sat_fat_g"]:
    data[f"{nutrient}_mean"] = data[[f"{nutrient}_day1", f"{nutrient}_day2"]].mean(axis=1)
data["na_k_ratio_mean"] = na_k_ratio(data["sodium_mg_mean"], data["potassium_mg_mean"])

## 6. Feature sets and missing values

Three feature sets are defined for comparison: demographic only, demographic with day 1 diet, and demographic with multi-day diet. A check confirms that no blood pressure variable is used as a feature.

Participants with missing values in the demographic features, the day 1 dietary features or the label are excluded. Missing day 2 values are expected and are not excluded.

In [7]:
DEMOGRAPHIC = ["age", "sex_female", "bmi"]
DIET_DAY1 = ["sodium_mg_day1", "potassium_mg_day1", "energy_kcal_day1",
             "sat_fat_g_day1", "na_k_ratio_day1"]
DIET_MULTIDAY = ["sodium_mg_mean", "potassium_mg_mean", "energy_kcal_mean",
                 "sat_fat_g_mean", "na_k_ratio_mean"]
TARGET = "bp_category"

bp_columns = SBP_COLS + DBP_COLS + ["systolic_avg", "diastolic_avg"]
assert not set(DEMOGRAPHIC + DIET_DAY1 + DIET_MULTIDAY) & set(bp_columns), "BP variable used as feature"

required = DEMOGRAPHIC + DIET_DAY1 + [TARGET]
print("Missing values before exclusion:")
print(data[required + ["sodium_mg_day2"]].isna().sum().to_string())

data = data.dropna(subset=required).copy()
print(f"\nParticipants after exclusion: {len(data)}")

Missing values before exclusion:
age                     0
sex_female              0
bmi                    56
sodium_mg_day1          0
potassium_mg_day1       0
energy_kcal_day1        0
sat_fat_g_day1          0
na_k_ratio_day1         0
bp_category             0
sodium_mg_day2       1017

Participants after exclusion: 7473


## 7. Output

The prepared dataset is saved to `data/processed/`. The averaged blood pressure columns are retained for reference only and are not model inputs.

In [8]:
output_cols = (["participant_id", "sex"] + DEMOGRAPHIC + DIET_DAY1 + DIET_MULTIDAY
               + [TARGET, "systolic_avg", "diastolic_avg"])
final = data[output_cols].copy()
final["participant_id"] = final["participant_id"].astype(int)

final.to_csv(PROCESSED_DIR / "nhanes_bp_diet.csv", index=False)
print(f"Saved {final.shape[0]} rows x {final.shape[1]} columns")
print(final[TARGET].value_counts().reindex(categories).to_string())

Saved 7473 rows x 18 columns
bp_category
Normal      3279
Elevated    1053
Stage 1     1554
Stage 2     1587
